In [1]:
import spacy


In [2]:
corpus = ["Thor ate pizza",
          "Loki is tall",
          "Loki is eating pizze"]
          

In [3]:
nlp = spacy.load("en_core_web_sm")

In [4]:
def preprocessing(text):
    doc = nlp(text)
    filtered_token =[]
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        filtered_token.append(token.lemma_)
    return (" ").join(filtered_token)

In [5]:
preprocessing("I am eating the pizzsa ")

'eat pizzsa'

In [6]:
processed_cropus = [preprocessing(text) for text in corpus]
processed_cropus

['thor eat pizza', 'Loki tall', 'Loki eat pizze']

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
v = CountVectorizer(ngram_range=(1,2))
v.fit(processed_cropus)
v.vocabulary_


{'thor': 9,
 'eat': 0,
 'pizza': 6,
 'thor eat': 10,
 'eat pizza': 1,
 'loki': 3,
 'tall': 8,
 'loki tall': 5,
 'pizze': 7,
 'loki eat': 4,
 'eat pizze': 2}

In [8]:
v.transform(["Thor ate pizza"]).toarray()

array([[0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0]])

In [9]:
import pandas as pd
df =  pd.read_json('news_dataset.json')

In [10]:
df.head()

,text,category
0,Watching Schrödinger's Cat Die University of C...,SCIENCE
1,WATCH: Freaky Vortex Opens Up In Flooded Lake,SCIENCE
2,Entrepreneurs Today Don't Need a Big Budget to...,BUSINESS
3,These Roads Could Recharge Your Electric Car A...,BUSINESS
4,Civilian 'Guard' Fires Gun While 'Protecting' ...,CRIME


In [11]:
df.category.value_counts()

category
BUSINESS    4254
SPORTS      4167
CRIME       2893
SCIENCE     1381
Name: count, dtype: int64

In [12]:
min_sample = 1381
df_business = df[df.category == 'BUSINESS'].sample(min_sample,random_state=42)
df_sports = df[df.category == 'SPORTS'].sample(min_sample,random_state=42)
df_crime = df[df.category == 'CRIME'].sample(min_sample,random_state=42)
df_science = df[df.category == 'SCIENCE'].sample(min_sample,random_state=42)


In [13]:
df_balanced = pd.concat([df_business,df_sports,df_crime,df_science],axis=0)
df_balanced.category.value_counts()

category
BUSINESS    1381
SPORTS      1381
CRIME       1381
SCIENCE     1381
Name: count, dtype: int64

In [14]:
df_balanced['category_num'] = df_balanced.category.map({
    'BUSINESS':0,
    'SPORTS':1,
    'CRIME':2,
    'SCIENCE':3
})

In [15]:
df_balanced.head()

,text,category,category_num
594,How to Develop the Next Generation of Innovato...,BUSINESS,0
3093,"Madoff Victims' Payout Nears $7.2 Billion, Tru...",BUSINESS,0
7447,Bay Area Floats 'Sanctuary In Transit Policy' ...,BUSINESS,0
10388,Microsoft Agrees To Acquire LinkedIn For $26.2...,BUSINESS,0
1782,"Inside A Legal, Multibillion Dollar Weed Market",BUSINESS,0


In [16]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(df_balanced.text,df_balanced.category_num,test_size=0.2,random_state=42,stratify = df_balanced.category_num)

In [17]:
y_test.value_counts()

category_num
2    277
0    276
1    276
3    276
Name: count, dtype: int64

In [18]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

clf = Pipeline([
    ('vectorizer',CountVectorizer()),
    ('Multi nb', MultinomialNB())
])

clf.fit(x_train,y_train)
y_pred= clf.predict(x_test)


In [19]:
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

           0       0.92      0.78      0.84       327
           1       0.85      0.92      0.88       256
           2       0.89      0.91      0.90       269
           3       0.82      0.89      0.85       253

    accuracy                           0.87      1105
   macro avg       0.87      0.88      0.87      1105
weighted avg       0.87      0.87      0.87      1105



In [21]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

clf = Pipeline([
    ('vectorizer',CountVectorizer(ngram_range=(1,2))),
    ('Multi nb', MultinomialNB())
])

clf.fit(x_train,y_train)
y_pred= clf.predict(x_test)

print(classification_report(y_pred,y_test))


              precision    recall  f1-score   support

           0       0.95      0.73      0.82       357
           1       0.83      0.92      0.87       247
           2       0.88      0.91      0.89       268
           3       0.79      0.93      0.85       233

    accuracy                           0.86      1105
   macro avg       0.86      0.87      0.86      1105
weighted avg       0.87      0.86      0.86      1105

